# Auto-RAN Blueprint

## Prerequisites & Environment setup

Install **docker** and **docker compose** if they are missing.

In [ ]:
!command -v docker >/dev/null 2>&1 || (curl -fsSL https://get.docker.com -o get-docker.sh && sudo sh get-docker.sh)
!docker compose version >/dev/null 2>&1 || sudo apt-get install -y docker-compose-plugin

Install Python dependencies.

In [ ]:
%pip install requests

Agents need some environment variables to work properly. The following script helps you generating a proper `.env` file for each agent.

Please, paste your Nvidia API Key in the following block of code before running it.

In [ ]:
api_key = ''

Run the following block to setup required environment variables for the agents.

**Note**: the script sets both the `API_KEY` and `NVIDIA_API_KEY` environment variables because the toolkits used for development (BAT-ADK and NAT) expect the `API_KEY` variable to be set, while the Nvidia model expects the `NVIDIA_API_KEY` variable to be set.

In [ ]:
from pathlib import Path

agents = {
    "config_planner": 9201,
    "monitoring": 9202,
    "validation": 9203,
}

template = """URL=http://localhost/
PORT={port}
MODEL=nvidia:meta/llama-3.3-70b-instruct
API_KEY={api_key}
NVIDIA_API_KEY={api_key}
"""

overwrite = True  # set to True to overwrite existing .env files

for agent, port in agents.items():
    path = Path(f"./agents/{agent}")
    path.mkdir(parents=True, exist_ok=True)

    env_file = path / ".env"
    if not env_file.exists() or overwrite:  # only write if it doesn't exist or overwrite is True
        env_file.write_text(template.format(port=port, api_key=api_key))
        print(f"✅ Created {env_file}")
    else:
        print(f"⚠️ {env_file} already exists, skipping.")

print("✅ Done")

## Network Overview

This notebook connects to an emulated OAI 5G Network (Physical Network) running on the public BubbleRAN server.

The network deployment consists of:
- An OAI **core network** (AMF, DB, SMF, UPF)
- An OAI **RAN** (gNB)
- A Near RT RIC, **FlexRIC**
- A **monitoring xApp** to collect real time KPIs
- An RF-sim **UE**

The agents will also deploy a Digital Twin Network with the same components of the Physical Network.

**Disclaimer**: _As this is a fully emulated environment, the BubbleRAN server assigns extra resources to the Digital Twin Network to emulate improved configurations. This setup is not intended for accurate performance measurements._

## Build and run the agents

Run `docker compose build` to build the agents container images.

In [ ]:
!docker compose build

Inizialize a session with the BubbleRAN public server (the request takes around 30s to run).

**NOTE**: Your session will be active for 30 minutes. Please, **CLEAN UP** the session once you are done with the blueprint!

In [ ]:
!curl -X POST http://193.55.113.25:8080/init

When the previous cell succeedes, the `colsstb01` 5G Network becomes available for you on BubbleRAN's public server. Since the server can potentially host multiple 5G Networks in parallel, you should always specify the network name to the Agents when sending a request, otherwise the behavior of the agents is undefined.

At this point you can start using the agents.

**NOTE**: if the initialization request did not succeed, the agents won't be able to run properly and will return the following error:
```
An error occurred while processing your request: Error in LangGraph workflow: unhandled errors in a TaskGroup (1 sub-exception)
```
Do not proceed if the `curl` request returned an error!

Start the agents containers in **detached** mode (`-d`).

In [ ]:
!docker compose up -d

Check which containers are active with `docker ps`.

In [ ]:
!docker ps

## Use the agents

The following block of code defines a function which sends requests to the **Config Planner**. Please, run it to be able to easily send requests to the agent.

You may need to change the `context_id` parameter when you call this function for multiple times.

In [ ]:
import requests
import json

context_id = 1

def stream_config_planner(user_text: str):
    global context_id
    headers = {
        "Content-Type": "application/json",
    }
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "message/stream",
        "params": {
            "configuration": {"accepted_output_modes": ["text"]},
            "message": {
                "context_id": f"{context_id}",
                "message_id": "1",
                "role": "user",
                "parts": [{"type": "text", "text": user_text}]
            }
        }
    }
    context_id += 1
    last_text = None
    config_planner_url = "http://localhost:9201"

    with requests.post(config_planner_url, headers=headers, json=payload, stream=True) as response:
        if response.status_code != 200:
            print("Request failed:", response.status_code, response.text)
            return

        for line in response.iter_lines():
            if not line:
                continue
            try:
                text = line.decode("utf-8")
                if text.startswith("data:"):
                    text = text[len("data:"):].strip()
                data = json.loads(text)
                result = data.get("result", {})

                if "status" in result:
                    message = result["status"].get("message", {})
                    for part in message.get("parts", []):
                        if part.get("kind") == "text" and part["text"] != last_text:
                            print("> " + part["text"])
                            last_text = part["text"]

                # Artifact updates (final output)
                elif "artifact" in result:
                    for part in result["artifact"].get("parts", []):
                        if part.get("kind") == "text" and part["text"] != last_text:
                            print("\n--- FINAL ANSWER ---\n")
                            print(part["text"])
                            last_text = part["text"]

            except json.JSONDecodeError:
                continue

Send a monitoring request to the **Config Planner**.
- The request will be forwarded to the **Monitoring Agent**.

In [ ]:
stream_config_planner(
    user_text="What are the current P0 nominal and uplink throughput of the 'colsstb01' network?"
)

If you see that the agent is not able to provide you a value for the throughput, it means that the UE was still not able to start traffic generation. Please re-run the query after some time (20-30 seconds).

Send an optimization request to the **Config Planner**.
- The Config Planner will use the **Monitoring Agent** to retrieve live KPIs and configurations
- It will run an optimization algorithm to propose a new P0 Nominal candidate
- The candidate will be validated in a **Network Digital Twin** environment using the **Validation Agent**
- Based on the validation results, the **Config Planner** decides if it can be beneficial to enforce the new configuration or not

Please keep in mind that the deployment of the **Network Digital Twin** and the traffic simulation can take several seconds (around 30 seconds for the deployment, and 30 seconds for the simulation).

The system is not frozen, if something goes wrong the server will return a Timeout Error after 180 seconds!

In [ ]:
stream_config_planner(
    user_text="Please optimize the P0 nominal of the 'colsstb01' network to improve uplink throughput"
)

Send a new monitoring request to the **Config Planner** (same as before)
- The request will be forwarded to the **Monitoring Agent**.

In [ ]:
stream_config_planner(
    user_text="What are the current P0 nominal and uplink throughput of the 'colsstb01' network?"
)

At this point:
- If no enforcement was performed, you can try to re-run the optimization query
- If the enforcement was performed, you can try to run a new monitoring query to double check that the P0 Nominal of the network has been changed. However, it's very likely that you won't see any difference in the uplink throughput because the environment is emulated.

**Note**: _If you notice a mismatch between the throughput values reported by the **Config Planner** during validation and those reported when executing the monitoring query, that's because the validation is performed based on the throughput values measured in the twin, which may be different from those measured in the physical network.

## Stop the containers and clean-up the resources

Run the following cell to release your session and allow other people to try this blueprint! :)

In [ ]:
!curl -X POST http://193.55.113.25:8080/cleanup

Stop the containers.

In [ ]:
!docker compose down